# Modelos, alvos e features de textoEquivale aos scripts `00`, `03_regressao.sql`, `04_classificacao.sql` e `05_features_texto.sql`.---**PDM 2026.2 — aula de 04/09/2026 · notebook 2 de 3**Este notebook e o mesmo conteudo dos scripts `.sql` da aula, so que rodando de dentrodo Python. Serve para estudar sozinho, para repetir a aula no seu ritmo e como pontepara a proxima aula, em que o dado sai do BigQuery e vira DataFrame.Voce pode rodar tudo no [Google Colab](https://colab.research.google.com) sem instalar nada.Se rodar na sua maquina, precisa de `pip install google-cloud-bigquery pandas db-dtypes`e de um `gcloud auth application-default login` antes.Uma coisa de cada vez: **nao rode tudo de uma vez**. Rode uma celula, leia o resultado,so entao passe para a proxima. O valor da aula esta em olhar para o numero que aparece.

In [ ]:
# Rode este bloco UMA VEZ, antes de qualquer outro.# Ele autentica voce no Google Cloud e cria os atalhos q(...) e run(...).PROJETO = "SEU_PROJETO"      # <<< troque pelo id do SEU projeto no GCPDATASET = "anuncios"LOCAL   = "us-central1"      # a regiao do dataset. Precisa bater, senao o BigQuery recusa.try:    from google.colab import auth    auth.authenticate_user()          # Colab: abre a janela de login do Googleexcept ImportError:    pass                              # local: rode antes `gcloud auth application-default login`import pandas as pdfrom google.cloud import bigqueryclient = bigquery.Client(project=PROJETO, location=LOCAL)def q(sql: str) -> pd.DataFrame:    """Roda o SQL e devolve um DataFrame. Toda ocorrencia de SEU_PROJETO vira o seu projeto."""    return client.query(sql.replace("SEU_PROJETO", PROJETO)).to_dataframe()def run(sql: str) -> None:    """Para CREATE TABLE / CREATE MODEL: executa, nao devolve tabela."""    client.query(sql.replace("SEU_PROJETO", PROJETO)).result()    print("ok")pd.set_option("display.float_format", lambda v: f"{v:,.2f}")print("conectado em", PROJETO, "|", LOCAL)

## 1. O modelo cego (v0)Na aula anterior voce treinou um modelo de preco sobre o dado **como ele estava**,sem nenhuma limpeza. Ele se chama `modelo_preco_imoveis`. Antes de tocar nele,olhe o que ele entregou.

In [ ]:
q("""SELECT *FROM ML.EVALUATE(MODEL `SEU_PROJETO.anuncios.modelo_preco_imoveis`)""")

**Como ler.** Duas colunas importam.- `mean_absolute_error` (MAE): de quanto o modelo erra, em reais, no anuncio tipico.- `r2_score`: quanto da variacao do preco o modelo explica. Zero significa "tao bom  quanto chutar sempre a media". **Negativo significa pior que chutar a media.**Um R2 negativo nao e um modelo ruim: e um modelo que atrapalha. Voce estaria melhorrespondendo "o preco medio" para qualquer imovel que te perguntassem.Ninguem errou o `CREATE MODEL`. O algoritmo fez exatamente o que foi pedido, sobreum dado em que 3% das linhas eram impossiveis.---## 2. Alavanca 1 — limpar o dado (v1)Mesmo algoritmo, mesmas colunas, mesma sintaxe. So muda a tabela de origem:em vez da silver crua, a `anuncios_gold` do notebook 1.

In [ ]:
run("""CREATE OR REPLACE MODEL `SEU_PROJETO.anuncios.modelo_preco`OPTIONS (  model_type            = 'LINEAR_REG',  input_label_cols      = ['preco'],  data_split_method     = 'AUTO_SPLIT',  enable_global_explain = TRUE) ASSELECT  preco,  area_util, quartos, banheiros, suites, garagem, condominio, iptu,  bairro,  eh_comercialFROM `SEU_PROJETO.anuncios.anuncios_gold`""")

**Como ler o OPTIONS.** Sao quatro linhas e cada uma decide algo:- `model_type` — qual familia de algoritmo. Trocar essa string troca o modelo inteiro.- `input_label_cols` — **o que voce quer prever**. Tudo que sobra no SELECT vira feature.- `data_split_method='AUTO_SPLIT'` — o BigQuery separa treino e validacao sozinho.  Sem isso voce avalia o modelo nos mesmos dados em que ele treinou, e o numero mente.- `enable_global_explain` — liga a explicabilidade, usada mais abaixo.Repare que `bairro` e texto e entra assim mesmo: o BQML faz o one-hot sozinho.

In [ ]:
q("""SELECT *FROM ML.EVALUATE(MODEL `SEU_PROJETO.anuncios.modelo_preco`)""")

**Como ler.** Compare com o v0. O R2 saiu do negativo e o MAE caiu varias vezes.Nada de ML mudou entre os dois modelos: o que mudou foi um `WHERE` e alguns `CASE WHEN`.Essa e a primeira das tres alavancas da aula, e e a que menos parece ML.Onde ele ainda erra mais?

In [ ]:
q("""SELECT  bairro, eh_comercial, area_util, quartos,  ROUND(preco)                   AS preco_real,  ROUND(predicted_preco)         AS preco_previsto,  ROUND(predicted_preco - preco) AS erroFROM ML.PREDICT(  MODEL `SEU_PROJETO.anuncios.modelo_preco`,  (SELECT * FROM `SEU_PROJETO.anuncios.anuncios_gold`))ORDER BY ABS(predicted_preco - preco) DESCLIMIT 20""")

**Como ler.** Erro grande nao e aleatorio, tem padrao. Procure o que os piores casostem em comum: bairro, faixa de area, tipo de imovel. E ai que mora a proxima feature.

In [ ]:
q("""SELECT *FROM ML.GLOBAL_EXPLAIN(MODEL `SEU_PROJETO.anuncios.modelo_preco`)ORDER BY attribution DESC""")

**Como ler.** `attribution` e quanto cada coluna pesou na decisao do modelo, em media.Nao e causalidade: e o quanto o modelo se apoiou naquela coluna. Serve para dois usosopostos — descobrir o que importa, e desconfiar quando uma coluna pesa demais(volte a isso no notebook 3).Agora use o modelo em um imovel que nao existe na tabela.

In [ ]:
q("""SELECT ROUND(predicted_preco) AS preco_estimadoFROM ML.PREDICT(  MODEL `SEU_PROJETO.anuncios.modelo_preco`,  (SELECT     180            AS area_util,     3              AS quartos,     2              AS banheiros,     1              AS suites,     2              AS garagem,     800.0          AS condominio,     2400.0         AS iptu,     'Setor Bueno'  AS bairro,     FALSE          AS eh_comercial  ))""")

> Cuidado com o tipo: `180` e INT64 como na silver. Escrever `180.0` daria erro de> coercao, porque o BigQuery nao converte FLOAT64 para INT64 sozinho aqui.---## 3. Trocar o alvo — classificacaoAte agora o alvo era um numero. Troque por um `BOOL` e o mesmo dado responde outrapergunta: **este anuncio e comercial ou residencial?**Antes do modelo, o baseline. Sempre.

In [ ]:
q("""SELECT  COUNT(*)                                             AS total,  COUNTIF(NOT eh_comercial)                            AS residenciais,  ROUND(100 * COUNTIF(NOT eh_comercial) / COUNT(*), 1) AS acuracia_do_chuteFROM `SEU_PROJETO.anuncios.anuncios_gold`""")

**Como ler.** `acuracia_do_chute` e o que voce acerta respondendo "residencial" paratodo mundo, sem modelo nenhum. Qualquer acuracia abaixo disso e um modelo pior queuma resposta fixa. Este numero e a regua; guarde-o antes de treinar.

In [ ]:
run("""CREATE OR REPLACE MODEL `SEU_PROJETO.anuncios.modelo_comercial`OPTIONS (  model_type            = 'LOGISTIC_REG',  input_label_cols      = ['eh_comercial'],  data_split_method     = 'AUTO_SPLIT',  enable_global_explain = TRUE) ASSELECT  eh_comercial,  preco,  area_util, quartos, banheiros, suites, garagem, condominio, iptu,  bairroFROM `SEU_PROJETO.anuncios.anuncios_gold`""")

**Como ler.** Duas linhas mudaram em relacao ao modelo de preco: `model_type` e`input_label_cols`. Mais nada. E repare que agora `preco` esta do lado das features —o que era alvo virou entrada. Trocar a pergunta reorganiza a tabela inteira.

In [ ]:
q("""SELECT *FROM ML.EVALUATE(MODEL `SEU_PROJETO.anuncios.modelo_comercial`)""")

**Como ler.** Agora as metricas sao outras: `accuracy`, `precision`, `recall`, `f1_score`,`roc_auc`. Compare a acuracia com o baseline que voce guardou. Se o ganho for de poucospontos, o modelo esta aprendendo pouco alem do obvio.

In [ ]:
q("""SELECT *FROM ML.CONFUSION_MATRIX(MODEL `SEU_PROJETO.anuncios.modelo_comercial`)""")

**Como ler.** A acuracia e um numero so e esconde de que lado o modelo erra. A matrizmostra: quantos comerciais ele chamou de residenciais e vice-versa. Em problemadesbalanceado, quase todo o erro cai na classe rara.Os erros mais interessantes sao os que o modelo cometeu com confianca alta.

In [ ]:
q("""SELECT  bairro, area_util, quartos,  ROUND(preco)           AS preco,  eh_comercial           AS verdade,  predicted_eh_comercial AS palpite,  ROUND((SELECT p.prob FROM UNNEST(predicted_eh_comercial_probs) p         WHERE p.label = predicted_eh_comercial), 3) AS confiancaFROM ML.PREDICT(  MODEL `SEU_PROJETO.anuncios.modelo_comercial`,  (SELECT * FROM `SEU_PROJETO.anuncios.anuncios_gold`))WHERE predicted_eh_comercial != eh_comercialORDER BY confianca DESCLIMIT 15""")

**Como ler.** Erro com confianca alta e o pior tipo de erro: o modelo esta seguro eerrado. Olhe os titulos desses casos — muitos sao rotulo ruim, nao modelo ruim.O `eh_comercial` veio de um REGEXP que voce mesmo escreveu; ele tambem erra.---## 4. Alavanca 2 — features de texto (v3)O `titulo` e a coluna mais bagunçada da tabela e a unica que um humano escreveu.Ela sabe coisas que nenhuma coluna numerica sabe.Comece pela hipotese obvia: piscina encarece?

In [ ]:
q(r"""SELECT  COUNTIF(REGEXP_CONTAINS(LOWER(titulo), r'piscina'))                        AS anuncios_com_piscina,  ROUND(AVG(IF(REGEXP_CONTAINS(LOWER(titulo), r'piscina'), preco, NULL)))     AS preco_medio_com,  ROUND(AVG(IF(NOT REGEXP_CONTAINS(LOWER(titulo), r'piscina'), preco, NULL))) AS preco_medio_semFROM `SEU_PROJETO.anuncios.anuncios_gold`""")

**Como ler.** Olhe primeiro `anuncios_com_piscina`. Se a contagem for baixa, adiferenca de medias ao lado nao sustenta conclusao nenhuma — e ruido de amostrapequena. A hipotese mais obvia da aula morre aqui, e isso e um resultado, nao um fracasso.A que sobrevive e outra: **que tipo de imovel e este?**

In [ ]:
q(r"""SELECT  REGEXP_CONTAINS(LOWER(titulo), r'apartamento|apto|studio|kitnet|flat') AS eh_apartamento,  REGEXP_CONTAINS(LOWER(titulo), r'casa|sobrado|mans[aã]o')              AS eh_casa,  REGEXP_CONTAINS(LOWER(titulo), r'lote|terreno|[aá]rea |fazenda|ch[aá]cara|s[ií]tio|rural') AS eh_terreno,  COUNT(*)                                                    AS anuncios,  ROUND(APPROX_QUANTILES(preco, 100)[OFFSET(50)])             AS mediana_preco,  ROUND(APPROX_QUANTILES(preco / area_util, 100)[OFFSET(50)]) AS mediana_reais_por_m2FROM `SEU_PROJETO.anuncios.anuncios_gold`GROUP BY 1, 2, 3ORDER BY anuncios DESC""")

**Como ler.** Compare `mediana_reais_por_m2` entre as linhas. Um terreno e umapartamento com a mesma area custam precos de ordens diferentes por metro quadrado.A gold nao tem coluna de tipo de imovel — a silver perdeu essa informacao. O tituloainda tem.Isso e o que uma feature de texto faz: devolve ao modelo uma variavel que o schema perdeu.

In [ ]:
run(r"""CREATE OR REPLACE TABLE `SEU_PROJETO.anuncios.gold_texto` ASSELECT  *,  REGEXP_CONTAINS(LOWER(titulo), r'piscina')                               AS tem_piscina,  REGEXP_CONTAINS(LOWER(titulo), r'alto padr[aã]o|luxo|requintad|exclusiv') AS eh_alto_padrao,  REGEXP_CONTAINS(LOWER(titulo), r'mobiliad|armári|armari|planejad')       AS tem_mobilia,  REGEXP_CONTAINS(LOWER(titulo), r'lote|terreno|[aá]rea |fazenda|ch[aá]cara|s[ií]tio|rural') AS eh_terreno,  REGEXP_CONTAINS(LOWER(titulo), r'apartamento|apto|studio|kitnet|flat')   AS eh_apartamento,  REGEXP_CONTAINS(LOWER(titulo), r'casa|sobrado|mans[aã]o')                AS eh_casaFROM `SEU_PROJETO.anuncios.anuncios_gold`""")

In [ ]:
q("""SELECT  COUNT(*)                AS total,  COUNTIF(tem_piscina)    AS piscina,  COUNTIF(eh_alto_padrao) AS alto_padrao,  COUNTIF(tem_mobilia)    AS mobilia,  COUNTIF(eh_terreno)     AS terreno,  COUNTIF(eh_apartamento) AS apartamento,  COUNTIF(eh_casa)        AS casaFROM `SEU_PROJETO.anuncios.gold_texto`""")

**Como ler.** Cobertura importa mais que a ideia. Uma flag que aparece em 20 anunciosde 887 nao tem como mover a agulha, por melhor que seja a intuicao por tras dela.As flags que valem sao as de cobertura alta.

In [ ]:
run("""CREATE OR REPLACE MODEL `SEU_PROJETO.anuncios.modelo_preco_v3`OPTIONS (  model_type            = 'LINEAR_REG',  input_label_cols      = ['preco'],  data_split_method     = 'AUTO_SPLIT',  enable_global_explain = TRUE) ASSELECT  preco,  area_util, quartos, banheiros, suites, garagem, condominio, iptu,  bairro, eh_comercial,  tem_piscina, eh_alto_padrao, tem_mobilia,  eh_terreno, eh_apartamento, eh_casaFROM `SEU_PROJETO.anuncios.gold_texto`""")

**Como ler.** `model_type` e `input_label_cols` sao identicos ao v1. Nenhumhiperparametro mudou. A unica diferenca sao seis colunas novas — e todas elassairam de uma coluna de texto que ja estava na tabela desde o inicio.

In [ ]:
placar = q("""SELECT 'v0_dado_como_estava' AS modelo, mean_absolute_error, r2_scoreFROM ML.EVALUATE(MODEL `SEU_PROJETO.anuncios.modelo_preco_imoveis`)UNION ALLSELECT 'v1_gold_limpa', mean_absolute_error, r2_scoreFROM ML.EVALUATE(MODEL `SEU_PROJETO.anuncios.modelo_preco`)UNION ALLSELECT 'v3_gold_mais_texto', mean_absolute_error, r2_scoreFROM ML.EVALUATE(MODEL `SEU_PROJETO.anuncios.modelo_preco_v3`)ORDER BY r2_score""")placar

**Como ler.** Duas alavancas, dois saltos. Do v0 para o v1 foi limpeza de dado; do v1para o v3 foi engenharia de feature. Nenhuma das duas mexeu no algoritmo.Um grafico ajuda a enxergar a ordem de grandeza — e ja e a ponte para a proxima aula,onde o resultado do BigQuery vira DataFrame e o Python assume dali para frente.

In [ ]:
import matplotlib.pyplot as pltfig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))ax1.barh(placar["modelo"], placar["mean_absolute_error"] / 1e6, color="#c0392b")ax1.set_title("Erro medio absoluto (milhoes de R$)")ax1.set_xlabel("MAE em R$ milhoes")ax2.barh(placar["modelo"], placar["r2_score"], color="#27ae60")ax2.axvline(0, color="#333", linewidth=1)ax2.set_title("R2  (zero = chutar a media)")plt.tight_layout()plt.show()

In [ ]:
q("""SELECT *FROM ML.GLOBAL_EXPLAIN(MODEL `SEU_PROJETO.anuncios.modelo_preco_v3`)ORDER BY attribution DESC""")

**Como ler.** Veja onde as flags de texto caem no ranking. A que a turma apostararamente e a que aparece no topo. A explicabilidade serve exatamente para isso:desmentir a intuicao com o dado que voce mesmo produziu.---## O que fica- Limpar o dado tirou o R2 do negativo. Foi a alavanca mais barata e a que mais rendeu.- Trocar o alvo e trocar duas linhas do `OPTIONS`. O resto do SQL fica igual.- Sempre calcule o baseline antes de comemorar acuracia.- Feature de texto devolve ao modelo o que o schema perdeu.- Cobertura da flag pesa mais que a genialidade da hipotese.**Proximo notebook:** a mesma coluna `titulo`, a mesma funcao `REGEXP`, e um modeloque parece otimo e nao vale nada.